In [37]:
import pandas as pd
import numpy as np

from datasets import Dataset

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

import torch

In [38]:
df = pd.read_csv("../../data/processed/final_dataset.csv")

In [39]:
print(df.head())

print(df.columns)

print(df.shape)

print(df.isnull().sum())

                                                text  target
0  update config browser work new syntax config b...       5
1  xalanc 19 current build fedora core 3 two type...       0
2  problem add new post delete post trying add ne...       1
3  loghandler work globalconfiguration clientconf...       4
4  decoding service broken orgapacheaxistransport...       4
Index(['text', 'target'], dtype='str')
(1147328, 2)
text      18130
target        0
dtype: int64


In [40]:
df.rename(columns={"target":"labels"}, inplace=True)

In [41]:
df = df.dropna(subset=["text","labels"])

In [42]:
df["text"] = df["text"].astype(str)

In [43]:
print(df["labels"].unique())

print(df["labels"].nunique())

[ 5  0  1  4 13 15  6  3 14  2 12 10  9 11  8  7]
16


In [44]:
num_labels = df["labels"].nunique()

print(num_labels)

16


In [45]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["labels"]
)

In [46]:
train_dataset = Dataset.from_pandas(train_df)

test_dataset = Dataset.from_pandas(test_df)

In [47]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [48]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [49]:
train_dataset = train_dataset.map(tokenize, batched=True)

test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/903358 [00:00<?, ? examples/s]

Map:   0%|          | 0/225840 [00:00<?, ? examples/s]

In [50]:
train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "token_type_ids",
        "labels"
    ]
)

test_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "token_type_ids",
        "labels"
    ]
)

In [51]:
print(train_dataset[0])

{'labels': tensor(4), 'input_ids': tensor([  101,  3256,  4059,  5587,  4736,  3207, 26557,  3508,  8873, 21928,
         9398,  3686, 15442,  2794,  2951,  3972, 12870,  5371,   102,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0, 

In [52]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [53]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [54]:
training_args = TrainingArguments(
    output_dir="../../models/bert_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01
)

In [55]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

c:\Users\Ahana Arora\OneDrive\Pictures\Documents\AI-Bug-Severity-Predictor\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
